In [1]:
import pandas as pd
import duckdb

pd.set_option('display.max_columns', 50)

## Functions

In [2]:
import plotly.graph_objects as go

def plot_events_over_time(df, title=''):

    fig = go.Figure()

    for event_name in df["event_name"].unique():

        temp = (
            df[df["event_name"] == event_name]
            .sort_values("year")
        )

        fig.add_trace(
            go.Scatter(
                x=temp["year"],
                y=temp["num_competitions"],
                mode="lines+markers",
                name=event_name,
                line=dict(width=3),
                marker=dict(size=8),
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Ano: %{x}<br>"
                    "Competições: %{y}<extra></extra>"
                )
            )
        )

    total_df = (
        df[["year", "total_competitions"]]
        .drop_duplicates()
        .sort_values("year")
    )

    fig.add_trace(
        go.Scatter(
            x=total_df["year"],
            y=total_df["total_competitions"],
            mode="lines+markers",
            name="Total de Competições",
            line=dict(width=6),
            marker=dict(size=10),
            hovertemplate=(
                "<b>Total de Competições</b><br>"
                "Ano: %{x}<br>"
                "Total: %{y}<extra></extra>"
            )
        )
    )

    fig.update_layout(
        title=title,
        xaxis_title="Ano",
        yaxis_title="Número de Competições",
        hovermode="closest",
        template="plotly_white",
        legend_title="Modalidade",
        height=700,
        legend=dict(
            itemclick="toggle",
            itemdoubleclick="toggleothers"
        )
    )

    return fig

def plot_event_presence_pct(df, title=''):

    fig = go.Figure()

    for event_name in df["event_name"].unique():

        temp = (
            df[df["event_name"] == event_name]
            .sort_values("year")
        )

        fig.add_trace(
            go.Scatter(
                x=temp["year"],
                y=temp["presence_pct"] * 100,
                mode="lines+markers",
                name=event_name,
                line=dict(width=3),
                marker=dict(size=8),
                hovertemplate=(
                    "<b>%{fullData.name}</b><br>"
                    "Ano: %{x}<br>"
                    "Presença: %{y:.1f}%<extra></extra>"
                )
            )
        )

    fig.update_layout(
        title=title,
        xaxis_title="Ano",
        yaxis_title="% das Competições",
        hovermode="closest",
        template="plotly_white",
        legend_title="Modalidade",
        height=700,
        legend=dict(
            itemclick="toggle",
            itemdoubleclick="toggleothers"
        )
    )

    return fig

## Análise mundo

In [3]:
con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,1080
1,222,2x2x2 Cube,882
2,pyram,Pyraminx,773
3,444,4x4x4 Cube,769
4,skewb,Skewb,673
5,333oh,3x3x3 One-Handed,663
6,clock,Clock,632
7,555,5x5x5 Cube,594
8,minx,Megaminx,575
9,333bf,3x3x3 Blindfolded,571


In [4]:
df_camp_by_year = con.execute("""
SELECT
    c.year,
    COUNT(DISTINCT c.id) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
WHERE
    c.cancelled = 0
GROUP BY c.year
ORDER BY c.year
""").df()

In [5]:
df_final = pd.DataFrame()
for i in range(2003, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [6]:
df_final['presence_pct'] = df_final['num_competitions'] / df_final['total_competitions']
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [7]:
fig_absolute_world = plot_events_over_time(df_final, 'Quantidade de Competições por Modalidade no Mundo')
fig_absolute_world

In [8]:
fig_percentage_world = plot_event_presence_pct(df_final, 'Presença das Modalidades nas Competições Mundiais')
fig_percentage_world

## Brasil

In [9]:
con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.country_id = 'Brazil'
    AND c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,40
1,222,2x2x2 Cube,35
2,pyram,Pyraminx,32
3,clock,Clock,31
4,minx,Megaminx,31
5,444,4x4x4 Cube,29
6,333oh,3x3x3 One-Handed,27
7,skewb,Skewb,26
8,333bf,3x3x3 Blindfolded,24
9,555,5x5x5 Cube,22


In [10]:
df_camp_by_year = con.execute("""
SELECT
    c.year,
    COUNT(DISTINCT c.id) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
WHERE
    c.country_id = 'Brazil'
    AND c.cancelled = 0
GROUP BY c.year
ORDER BY c.year
""").df()

In [11]:
df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.country_id = 'Brazil'
        AND c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [12]:
df_final.head()

,event_id,event_name,num_competitions,year,total_competitions
0,333,3x3x3 Cube,1,2007,1
1,555,5x5x5 Cube,1,2007,1
2,333bf,3x3x3 Blindfolded,1,2007,1
3,333ft,3x3x3 With Feet,1,2007,1
4,222,2x2x2 Cube,1,2007,1


In [13]:
df_final['presence_pct'] = df_final['num_competitions'] / df_final['total_competitions']
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [14]:
fig_absolute_brazil = plot_events_over_time(df_final, 'Quantidade de Competições por Modalidade no Brasil')
fig_absolute_brazil

In [15]:
fig_percentage_brazil = plot_event_presence_pct(df_final, 'Presença das Modalidades nas Competições Brasileiras')
fig_percentage_brazil

## SP

In [16]:
con = duckdb.connect()

df = con.execute("""
SELECT
    e.id AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT c.id) AS num_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    c.country_id = 'Brazil'
    AND c.city_name LIKE '%, São Paulo'
    AND c.year = 2026
GROUP BY
    e.id,
    e.name
ORDER BY
    num_competitions DESC
""").df()

df

,event_id,event_name,num_competitions
0,333,3x3x3 Cube,10
1,clock,Clock,9
2,333bf,3x3x3 Blindfolded,8
3,pyram,Pyraminx,7
4,333oh,3x3x3 One-Handed,7
5,minx,Megaminx,7
6,777,7x7x7 Cube,6
7,skewb,Skewb,6
8,444,4x4x4 Cube,6
9,222,2x2x2 Cube,6


In [17]:
df_camp_by_year = con.execute("""
SELECT
    c.year,
    COUNT(DISTINCT c.id) AS total_competitions
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    ON r.competition_id = c.id
WHERE
    c.country_id = 'Brazil'
    AND c.city_name LIKE '%, São Paulo'
    AND c.cancelled = 0
GROUP BY c.year
ORDER BY c.year
""").df()

In [18]:
df_camp_by_year

,year,total_competitions
0,2007,1
1,2009,2
2,2010,2
3,2011,1
4,2012,3
5,2013,9
6,2014,9
7,2015,11
8,2016,9
9,2017,14


In [19]:
df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
    SELECT
        e.id AS event_id,
        e.name AS event_name,
        COUNT(DISTINCT c.id) AS num_competitions
    FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    JOIN read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
        ON r.competition_id = c.id
    JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
        ON e.id = r.event_id
    WHERE
        c.country_id = 'Brazil'
        AND c.city_name LIKE '%, São Paulo'
        AND c.year = {i}
    GROUP BY
        e.id,
        e.name
    ORDER BY
        num_competitions DESC
    """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [20]:
df_final['presence_pct'] = df_final['num_competitions'] / df_final['total_competitions']
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [21]:
fig_absolute_sp = plot_events_over_time(df_final, 'Quantidade de Competições por Modalidade Em São Paulo')
fig_absolute_sp

In [22]:
fig_percentage_sp = plot_event_presence_pct(df_final, 'Presença das Modalidades nas Competições Paulistas')
fig_percentage_sp

## Rounds

In [23]:
df_final = pd.DataFrame()
for i in range(2006, 2027):

    df = con.execute(f"""
        WITH rounds AS (

            SELECT DISTINCT
                competition_id,
                event_id,
                round_type_id,
                format_id

            FROM read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t')

        )

        SELECT
            e.id AS event_id,
            e.name AS event_name,

            SUM(
                CASE

                    WHEN e.id IN ('333fm', '333mbf')
                    THEN
                        CASE rounds.format_id
                            WHEN '1' THEN 1
                            WHEN '2' THEN 2
                            WHEN '3' THEN 3
                            ELSE 1
                        END

                    ELSE 1

                END
            ) AS competitive_units

        FROM rounds

        JOIN read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
            ON c.id = rounds.competition_id

        JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
            ON e.id = rounds.event_id

        WHERE
            c.country_id = 'Brazil'
            AND c.city_name LIKE '%, São Paulo'
            AND c.year = {i}

        GROUP BY
            e.id,
            e.name

        ORDER BY
            competitive_units DESC
        """).df()

    df['year'] = i

    df_final = pd.concat([df_final, df])

df_final = df_final.merge(df_camp_by_year, on='year', how='left')

In [24]:
df_final = df_final[~df_final['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_final.sort_values(by='event_name', inplace=True)

In [25]:
fig_rounds_sp = plot_events_over_time(df_final.rename(columns={'competitive_units': 'num_competitions'}), 'Quantidade de rounds por Modalidade Em São Paulo')
fig_rounds_sp

## Competidores

In [26]:
df_active = con.execute("""
SELECT
    c.year,
    e.id   AS event_id,
    e.name AS event_name,
    COUNT(DISTINCT r.person_id) AS num_active_competitors
FROM read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
JOIN read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    ON c.id = r.competition_id
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = r.event_id
WHERE
    r.best > 0
    AND c.year > 2000
GROUP BY
    c.year, e.id, e.name
ORDER BY
    c.year, num_active_competitors DESC
""").df()

In [27]:
df_active_total = con.execute("""
SELECT
    c.year,
    COUNT(DISTINCT r.person_id) AS total_active_competitors
FROM read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
JOIN read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
    ON c.id = r.competition_id
WHERE
    r.best > 0
    AND c.year > 2000
GROUP BY c.year
ORDER BY c.year
""").df()

df_active = df_active.merge(df_active_total, on='year', how='left')

In [28]:
df_active = df_active[~df_active['event_name'].isin(['3x3x3 Multi-Blind Old Style', 'Magic', 'Master Magic'])]
df_active.sort_values(by='event_name', inplace=True)

In [29]:
fig_active = plot_events_over_time(
df_active.rename(columns={
        'num_active_competitors': 'num_competitions',
        'total_active_competitors': 'total_competitions',
    }),
    'Competidores Ativos por Modalidade no Mundo'
)
fig_active

In [30]:
df_active['presence_pct'] = (df_active['num_active_competitors'] / df_active['total_active_competitors'])

In [31]:
fig_active_pct = plot_event_presence_pct(df_active, 'Competidores Ativos por Modalidade no Mundo (percentual)')
fig_active_pct

In [32]:
df_new = con.execute("""
WITH first_year AS (
    -- primeiro ano de cada pessoa em cada modalidade = estreia
    SELECT
        r.person_id,
        r.event_id,
        MIN(c.year) AS estreia
    FROM read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    JOIN read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
        ON r.competition_id = c.id
    WHERE c.cancelled = 0
    GROUP BY r.person_id, r.event_id
)
SELECT
    e.name        AS event_name,
    fy.estreia    AS year,
    COUNT(*)      AS new_competitors
FROM first_year fy
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e
    ON e.id = fy.event_id
WHERE
    fy.estreia > 2000
GROUP BY e.name, fy.estreia
ORDER BY year, event_name
""").df()



In [33]:
fig_new_by_event = go.Figure()
for event_name in sorted(df_new["event_name"].unique()):
    temp = df_new[df_new["event_name"] == event_name].sort_values("year")
    fig_new_by_event.add_trace(go.Scatter(
        x=temp["year"], y=temp["new_competitors"],
        mode="lines+markers", name=event_name,
    ))
fig_new_by_event.update_layout(
    title="Novos competidores por ano, por modalidade",
    xaxis_title="Ano", yaxis_title="Novos competidores (estreias)", height=700,
)
fig_new_by_event

In [34]:
df_renov = con.execute("""
WITH res AS (
    -- base: quem competiu em qual modalidade em qual ano
    SELECT r.person_id, r.event_id, c.year
    FROM read_csv_auto('./tsv/WCA_export_results.tsv', delim='\t') r
    JOIN read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
        ON r.competition_id = c.id
    WHERE c.cancelled = 0
),
ativos AS (   -- competidores ativos: distintos no ano
    SELECT event_id, year, COUNT(DISTINCT person_id) AS ativos
    FROM res GROUP BY event_id, year
),
estreia AS (  -- ano de estreia de cada pessoa na modalidade
    SELECT person_id, event_id, MIN(year) AS year
    FROM res GROUP BY person_id, event_id
),
novos AS (
    SELECT event_id, year, COUNT(*) AS novos
    FROM estreia GROUP BY event_id, year
)
SELECT
    e.name AS event_name,
    a.year,
    a.ativos,
    COALESCE(n.novos, 0) AS novos,
    100.0 * COALESCE(n.novos, 0) / a.ativos AS pct_renovacao
FROM ativos a
LEFT JOIN novos n ON n.event_id = a.event_id AND n.year = a.year
JOIN read_csv_auto('./tsv/WCA_export_events.tsv', delim='\t') e ON e.id = a.event_id
WHERE a.year > 2000
ORDER BY a.year, event_name
""").df()

In [35]:
import plotly.graph_objects as go

fig_renov = go.Figure()
for event_name in sorted(df_renov["event_name"].unique()):
    temp = df_renov[df_renov["event_name"] == event_name].sort_values("year")
    fig_renov.add_trace(go.Scatter(
        x=temp["year"], y=temp["pct_renovacao"],
        mode="lines+markers", name=event_name,
        hovertemplate="%{x}: %{y:.1f}% novos<br>ativos=%{customdata[0]}, novos=%{customdata[1]}<extra>"+event_name+"</extra>",
        customdata=temp[["ativos", "novos"]].values,
    ))
fig_renov.update_layout(
    title="Renovação por modalidade (% dos ativos que são novos)",
    xaxis_title="Ano", yaxis_title="% de competidores novos",
    yaxis_ticksuffix="%",
    height=700,
)
fig_renov

## Mapa de Cobertura (Brasil, 2024+)

Em vez de empilhar um círculo por competição (o que satura regiões densas como São Paulo),
desenho a **silhueta da união** dos círculos: monto um campo de "distância até a competição
mais próxima" numa grade e extraio o contorno no nível do raio. Assim a opacidade é uniforme
e nenhuma região fica apagada.

Ajuste `DEFAULT_RADIUS_KM` (ou use o slider no mapa) para ver o quão coberto o país fica.

In [36]:
import numpy as np
from scipy.spatial import cKDTree
import matplotlib
matplotlib.use("Agg")            # só usamos o contour para o cálculo, sem exibir
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# Parâmetros do mapa de cobertura
# ----------------------------------------------------------------------
RADIUS_OPTIONS_KM = [50, 100, 150, 200]   # raios disponíveis no slider
DEFAULT_RADIUS_KM = 100                    # raio exibido ao abrir o mapa
GRID_STEP_DEG     = 0.08                    # resolução da grade (~9 km); menor = contorno mais suave
EARTH_RADIUS_KM   = 6371.0

# ----------------------------------------------------------------------
# Competições no Brasil, 2024 em diante, não canceladas
# ----------------------------------------------------------------------
comps = con.execute("""
SELECT
    c.cell_name              AS name,
    c.year,
    c.latitude_microdegrees  / 1e6 AS lat,
    c.longitude_microdegrees / 1e6 AS lon
FROM read_csv_auto('./tsv/WCA_export_competitions.tsv', delim='\t') c
WHERE c.country_id = 'Brazil'
  AND c.cancelled = 0
  AND c.year >= 2024
""").df()


def latlon_to_xyz(lat, lon, R=EARTH_RADIUS_KM):
    """Projeta lat/lon em coordenadas cartesianas na esfera (para distância real)."""
    la, lo = np.radians(lat), np.radians(lon)
    return np.column_stack([
        R * np.cos(la) * np.cos(lo),
        R * np.cos(la) * np.sin(lo),
        R * np.sin(la),
    ])


# ----------------------------------------------------------------------
# Campo de distância: para cada ponto da grade, distância (km) até a
# competição mais próxima. O contorno nesse campo no nível `raio` é
# exatamente a silhueta da UNIÃO dos círculos daquele raio.
# ----------------------------------------------------------------------
pad  = max(RADIUS_OPTIONS_KM) / 111.0 + 1.0        # margem p/ os círculos fecharem dentro da grade
lons = np.arange(comps.lon.min() - pad, comps.lon.max() + pad, GRID_STEP_DEG)
lats = np.arange(comps.lat.min() - pad, comps.lat.max() + pad, GRID_STEP_DEG)
LON, LAT = np.meshgrid(lons, lats)

tree = cKDTree(latlon_to_xyz(comps["lat"].values, comps["lon"].values))
chord, _ = tree.query(latlon_to_xyz(LAT.ravel(), LON.ravel()), k=1)   # distância em corda (km)
arc  = 2 * EARTH_RADIUS_KM * np.arcsin(np.clip(chord / (2 * EARTH_RADIUS_KM), 0, 1))  # corda -> arco
DIST = arc.reshape(LAT.shape)


def union_segments(level_km):
    """Extrai os anéis (loops) do contorno da união dos círculos de raio `level_km`."""
    fig_tmp, ax = plt.subplots()
    cs = ax.contour(LON, LAT, DIST, levels=[level_km])
    segs = [np.asarray(s) for s in cs.allsegs[0]]
    plt.close(fig_tmp)
    return segs


def union_trace(level_km, visible):
    """Uma trace com a silhueta da união (anéis separados por None) e opacidade uniforme."""
    lat_all, lon_all = [], []
    for seg in union_segments(level_km):
        lon_all.extend(seg[:, 0].tolist() + [None])
        lat_all.extend(seg[:, 1].tolist() + [None])
    return go.Scattermap(
        lat=lat_all, lon=lon_all,
        mode="lines",
        fill="toself",
        fillcolor="rgba(0, 120, 200, 0.25)",
        line=dict(width=1.2, color="rgba(0, 90, 160, 0.9)"),
        name=f"Cobertura {level_km} km",
        hoverinfo="skip",
        visible=visible,
    )


# ----------------------------------------------------------------------
# Figura: uma silhueta de união por raio + marcadores das sedes.
# O slider apenas alterna qual raio fica visível.
# ----------------------------------------------------------------------
fig_coverage = go.Figure()

for r in RADIUS_OPTIONS_KM:
    fig_coverage.add_trace(union_trace(r, visible=(r == DEFAULT_RADIUS_KM)))

fig_coverage.add_trace(go.Scattermap(
    lat=comps["lat"], lon=comps["lon"],
    mode="markers",
    marker=dict(size=6, color="rgb(200, 30, 30)"),
    name="Competições",
    text=comps["name"] + " (" + comps["year"].astype(str) + ")",
    hovertemplate="%{text}<extra></extra>",
    visible=True,
))

steps = []
for i, r in enumerate(RADIUS_OPTIONS_KM):
    vis = [j == i for j in range(len(RADIUS_OPTIONS_KM))] + [True]  # marcadores sempre visíveis
    steps.append(dict(
        method="update",
        args=[{"visible": vis}],
        label=f"{r} km",
    ))

fig_coverage.update_layout(
    title=f"Cobertura de Competições no Brasil (2024+) — {len(comps)} competições",
    map=dict(
        style="open-street-map",
        center=dict(lat=-14.5, lon=-51.5),
        zoom=3.2,
    ),
    height=800,
    margin=dict(l=0, r=0, t=60, b=0),
    sliders=[dict(
        active=RADIUS_OPTIONS_KM.index(DEFAULT_RADIUS_KM),
        currentvalue=dict(prefix="Raio de cobertura: "),
        pad=dict(t=30),
        steps=steps,
    )],
)

fig_coverage

## HTML Final

In [37]:
import re
from plotly.io import to_html

# URL do bundle do Plotly na CDN (usada uma única vez na página final de abas)
_PLOTLY_CDN_RE = re.compile(
    r'<script[^>]*src="https://cdn\.plot\.ly/[^"]*"[^>]*>\s*</script>', re.I
)
# marcadores que delimitam apenas os gráficos dentro de cada página
_CHARTS_START = "<!--CHARTS:START-->"
_CHARTS_END   = "<!--CHARTS:END-->"


def _charts_html(figures):
    """Monta os blocos de gráficos. Só o 1º embute o Plotly (via CDN)."""
    blocks = []
    for i, fig in enumerate(figures):
        graph_html = to_html(
            fig,
            include_plotlyjs="cdn" if i == 0 else False,
            full_html=False,
        )
        title = fig.layout.title.text or f"Gráfico {i + 1}"
        blocks.append(
            f'<div class="chart"><h2>{title}</h2>{graph_html}</div>'
        )
    return "\n".join(blocks)


_PAGE_CSS = """
    body {
        font-family: Arial, sans-serif;
        max-width: 1400px;
        margin: auto;
        padding: 20px;
    }
    h1 { text-align: center; }
    .chart { margin-top: 50px; margin-bottom: 50px; }
"""


def export_dashboard(figures, output_file, page_title="Análise WCA"):
    """
    Gera uma página HTML autônoma (funciona sozinha) com a lista de figuras.
    Os gráficos ficam entre marcadores para poderem ser reaproveitados nas abas.
    """
    html = f"""<!DOCTYPE html>
<html lang="pt-br">
<head>
    <meta charset="utf-8">
    <title>{page_title}</title>
    <style>{_PAGE_CSS}</style>
</head>
<body>
    <h1>{page_title}</h1>
    <p>
        Análise baseada nos
        <a href="https://www.worldcubeassociation.org/export/results"
           target="_blank" rel="noopener noreferrer">dados públicos da WCA</a>.
    </p>
    {_CHARTS_START}
    {_charts_html(figures)}
    {_CHARTS_END}
</body>
</html>"""

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)
    return output_file


def build_tabbed_page(tabs, output_file="index.html", page_title="WCA Brasil — Análise"):
    """
    Une várias páginas HTML (geradas por export_dashboard) numa única página com abas.

    tabs: list[tuple[str, str]] -> [(rótulo_da_aba, caminho_do_html), ...]
    """
    nav, panels, plotly_tag = [], [], None

    for i, (label, path) in enumerate(tabs):
        with open(path, encoding="utf-8") as f:
            html = f.read()

        # extrai só os gráficos (entre os marcadores)
        m = re.search(re.escape(_CHARTS_START) + r"(.*?)" + re.escape(_CHARTS_END), html, re.S)
        content = m.group(1) if m else html

        # captura o Plotly da CDN uma vez e remove das abas (evita carregar várias vezes)
        found = _PLOTLY_CDN_RE.search(content)
        if found and plotly_tag is None:
            plotly_tag = found.group(0)
        content = _PLOTLY_CDN_RE.sub("", content)

        active = " active" if i == 0 else ""
        nav.append(
            f'<button class="tab-btn{active}" data-tab="tab{i}">{label}</button>'
        )
        panels.append(
            f'<section id="tab{i}" class="tab-panel{active}">{content}</section>'
        )

    if plotly_tag is None:
        plotly_tag = '<script src="https://cdn.plot.ly/plotly-3.0.1.min.js"></script>'

    tabs_css = """
    body { font-family: Arial, sans-serif; margin: 0; padding: 0; }
    h1 { text-align: center; margin: 20px 0 0; }
    .tabs {
        position: sticky; top: 0; z-index: 10;
        display: flex; justify-content: center; gap: 8px;
        background: #fff; padding: 14px; border-bottom: 1px solid #ddd;
    }
    .tab-btn {
        font-size: 16px; padding: 10px 22px; cursor: pointer;
        border: 1px solid #ccc; border-radius: 6px; background: #f5f5f5;
    }
    .tab-btn.active { background: #0078c8; color: #fff; border-color: #0078c8; }
    .tab-panel { display: none; max-width: 1400px; margin: auto; padding: 20px; }
    .tab-panel.active { display: block; }
    .chart { margin-top: 50px; margin-bottom: 50px; }
    """

    tabs_js = """
    document.querySelectorAll('.tab-btn').forEach(function (btn) {
        btn.addEventListener('click', function () {
            document.querySelectorAll('.tab-btn').forEach(b => b.classList.remove('active'));
            document.querySelectorAll('.tab-panel').forEach(p => p.classList.remove('active'));
            btn.classList.add('active');
            var panel = document.getElementById(btn.dataset.tab);
            panel.classList.add('active');
            // gráficos renderizados escondidos ficam com largura 0: força o redimensionamento
            panel.querySelectorAll('.js-plotly-plot').forEach(function (gd) {
                if (window.Plotly) window.Plotly.Plots.resize(gd);
            });
        });
    });
    """

    html = f"""<!DOCTYPE html>
<html lang="pt-br">
<head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <title>{page_title}</title>
    {plotly_tag}
    <style>{tabs_css}</style>
</head>
<body>
    <h1>{page_title}</h1>
    <nav class="tabs">{"".join(nav)}</nav>
    {"".join(panels)}
    <script>{tabs_js}</script>
</body>
</html>"""

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(html)
    return output_file

In [38]:
# Figuras de cada aba
competicoes_figs = [
    fig_absolute_world, fig_percentage_world,
    fig_absolute_brazil, fig_percentage_brazil,
    fig_absolute_sp, fig_percentage_sp,
    fig_rounds_sp,
    fig_coverage,
]

competidores_figs = [
    fig_active, fig_active_pct,
    fig_new_by_event,
    fig_renov,
]

# 1) Uma página HTML autônoma por tema
export_dashboard(competicoes_figs,  "competicoes.html",  page_title="WCA Brasil — Competições")
export_dashboard(competidores_figs, "competidores.html", page_title="WCA Brasil — Competidores")

# 2) Página final com abas (deploy no GitHub Pages via index.html)
build_tabbed_page(
    [
        ("Competições",  "competicoes.html"),
        ("Competidores", "competidores.html"),
    ],
    output_file="index.html",
    page_title="WCA Brasil — Análise",
)

'index.html'